<a href="https://colab.research.google.com/github/JuanZapa7a/Medical-Image-Processing/blob/main/PIM_Challenge/PIM_Challenge_Student_Practice_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UPCT Medical Image Segmentation Challenge 2026-27
## Practice 9

**Course:** Medical Image Processing (521104007)

**Professor:** Juan Zapata

> **New** content of this practice. Copy the cells below and paste them **at the end** of your own notebook (the one you started in Practice 6) — do not repeat the previous practices, you already did them there.

## Session Guide (2 hours per session)
| Practice | Dates (Group A / B) | Session Objective | Visual Checkpoint |
|----------|----------------------|-----------------------|-------------------|
| **P6** | 28 Oct - 2 Nov | EDA, Dataset and RLE format | 6 images with masks + RLE OK |
| **P7** | 9-11 Nov | Baseline U-Net and 1st Submission | Loss plots + Kaggle Submission |
| **P8** | 16-18 Nov | Data Augmentation and improvement | Baseline vs Augmented comparison |
| ▶ **P9** | 23-25 Nov | Inference, Threshold and Errors | 5 normal images + 2 error cases |
| **P10** | 30 Nov-2 Dec | TTA, Final Submission and Defense | Best Dice Score + Oral Defense |

> **Golden Rule:** According to Art. 7.5 of the UPCT Assessment Regulations, attendance and validation of the Checkpoint in the classroom is mandatory to pass the practice.


# Practice 9: Inference, Threshold Optimization and Error Analysis
## Single session (23 Nov Group A / 25 Nov Group B)

### Session objectives:
1. Understand why the decision threshold of 0.5 is not always the optimal one in medicine.
2. Compute the **Optimal Threshold** by evaluating the validation set.
3. Perform a **Qualitative Error Analysis** (False Positives and False Negatives).
4. Generate the **Final Submission** for Kaggle.

###  Clinical Context: The FP vs FN dilemma
In breast cancer detection:
*   **False Negative (FN):** Saying there is no tumor when there actually is one. (Serious: delay in treatment).
*   **False Positive (FP):** Saying there is a tumor when there is none. (Anxiety, unnecessary biopsies).

Our model must find the perfect balance. Today we are going to look for it.

> **CHECKPOINT P9:** Show the professor:
> 1. Threshold vs Dice Score plot with the optimal point marked.
> 2. Visualization of 5 critical cases (including at least 1 "normal" image and 1 serious error).

## Before starting: choose the final model (P7 baseline vs P8 augmented)

Block 9.3 talks about the "final model": the one with the best Val Dice between the baseline (P7) and the augmented (P8). The following cell loads **from Drive** the checkpoints of both, compares their Val Dice, and leaves the winner in the `model` variable — so you do not have to decide by hand or risk accidentally using the wrong one.


In [ ]:
# ============================================================
# Choose the "final model": the best Val Dice between baseline and augmented
# ============================================================
paths = {
    'baseline (P7)': CHECKPOINT_DIR / 'baseline_checkpoint.pth',
    'augmented (P8)': CHECKPOINT_DIR / 'augmented_checkpoint.pth',
}

candidates = {}
for name, path in paths.items():
    if path.exists():
        ckpt = torch.load(path, map_location=DEVICE)
        candidates[name] = ckpt
        print(f"{name}: Val Dice = {ckpt['best_val_dice']:.4f}")
    else:
        print(f"{name}: not found in Drive ({path})")

if not candidates:
    raise FileNotFoundError("No checkpoint found. Complete P7 and/or P8 first.")

best_name = max(candidates, key=lambda n: candidates[n]['best_val_dice'])
best_checkpoint = candidates[best_name]

model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1, activation=None)
model = model.to(DEVICE)
model.load_state_dict(best_checkpoint['model_state_dict'])

print(f"\nFinal model: {best_name} (Val Dice = {best_checkpoint['best_val_dice']:.4f})")


## Block 9.1: The Threshold as a Hyperparameter
### 0.5 was never an informed decision

Since Practice 7 you have been using `probs > 0.5` to convert probabilities into a binary mask. That 0.5 did not come from any analysis of your data: it is simply the midpoint of the `sigmoid()` range (which goes from 0 to 1), the most "neutral" default option possible. Nothing guarantees that it is the best cut for this specific dataset.

> **Key idea:** the threshold is one more hyperparameter of your system, just like `lr` or `NUM_EPOCHS` — you had simply left it fixed without questioning it until now.

### Why the threshold is not free: precision vs recall

Remember the clinical dilemma from the introduction of this practice: a False Negative (not detecting a real tumor) and a False Positive (detecting a tumor that does not exist) do not have the same cost. The threshold directly controls that balance:

| Threshold | Effect on the mask | Consequence |
|-----------|------------------------|----------------|
| Lower (e.g. 0.3) | The model marks "tumor" with less evidence | Fewer False Negatives, more False Positives |
| Higher (e.g. 0.7) | The model requires more confidence to mark "tumor" | Fewer False Positives, more False Negatives |

The Dice Score you are going to maximize is a metric that weights precision and recall equally — it does not distinguish whether you err by FP or by FN. Maximizing Dice gives you the best *statistical* balance, but not necessarily the best *clinical* balance (where an FN is usually more serious than an FP).

> **Question to think about:** if you were the clinical decision-makers of this system and had to choose between the threshold that maximizes Dice and a slightly different one that reduces False Negatives at the cost of some Dice, which one would you choose? Why does the exercise, even so, ask you to maximize Dice?

### Why it is searched in validation, not in train or test

This is exactly the same principle of Practices 7 and 8 applied to a new hyperparameter:

| Set | Why NOT use it to choose the threshold? |
|----------|-------------------------------------------------|
| Train | The model already saw these data while training; the threshold would be fitted to the specific noise of those images |
| Test | It has no masks — you cannot compute Dice there. And even if it had them, choosing the threshold by looking at the test would be "cheating": you would be tuning a hyperparameter with the same data you are later evaluated on |
| Val | Data the model did not use to adjust weights, and you do have masks to measure Dice — the fair choice |

### Why there is no need to retrain to try each threshold

Unlike changing `lr` or adding augmentation (which force you to train the whole model again), the threshold is applied **after** the model has already produced its probabilities. The model weights do not change between one threshold and another — only where you cut those probabilities to decide 0 or 1 changes. That is why you can sweep 17 threshold values (0.1 to 0.9 in steps of 0.05) evaluating the `val_loader` once for each, with no training cost at all.

### How to interpret the Threshold vs Dice curve

The typical shape of this curve is a bell: low Dice at the extremes (very low thresholds generate too many False Positives; very high ones, too many False Negatives) and a maximum at some intermediate point.

| Curve shape | Interpretation |
|----------------------|-------------------|
| Marked and narrow peak | The threshold matters a lot; choosing badly costs Dice noticeably |
| Flat curve in the central zone | The model is robust to the exact threshold; several nearby values give a similar Dice |

### Quick summary

| Concept | Main idea |
|----------|-----------------|
| Threshold | One more hyperparameter, not a fixed constant |
| Precision vs recall | Low threshold → fewer FN, more FP; high threshold → the opposite |
| Where to search | In validation: train biases the tuning, test would invalidate the evaluation |
| Search cost | None in training — the model does not change, only the decision cut |
| Curve shape | A marked peak indicates that the threshold matters; a flat curve indicates that the model is robust over a range of values |

## Task 9.1: Optimal Threshold Search
So far we have used `threshold = 0.5` to binarize the probabilities. But is it really the best value for our dataset?

### Instructions:
1. Put the model in evaluation mode (`model.eval()`).
2. Iterate over the `val_loader` (without computing gradients).
3. For each batch, obtain the probabilities (`torch.sigmoid(logits)`).
4. Try a range of thresholds from **0.1 to 0.9** (in steps of 0.05).
5. For each threshold, compute the **average Dice Score** over the whole validation set.
6. Plot: **X axis** = Threshold, **Y axis** = Dice Score.
7. Identify and save the `best_threshold` that maximizes Dice.

> **Hint:** Do not modify the model weights. You are only evaluating how the metric changes when the decision cut changes.

In [ ]:
# ============================================================
# TASK 9.1: OPTIMAL THRESHOLD SEARCH
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

# WRITE YOUR CODE HERE
# 1. Define a list of thresholds to try: thresholds = np.arange(0.1, 0.95, 0.05)
# 2. Initialize a list to store the dices: val_dices_per_threshold = []
# 3. model.eval()
# 4. Loop for threshold in thresholds:
#       epoch_dice = 0
#       with torch.no_grad():
#           for images, masks in val_loader:
#               images, masks = images.to(DEVICE), masks.to(DEVICE)
#               logits = model(images)
#               probs = torch.sigmoid(logits)
#               preds = (probs > threshold).float() # <-- Here you use the threshold of the loop
#               # Compute Dice...
#               epoch_dice += dice.item()
#       val_dices_per_threshold.append(epoch_dice / len(val_loader))
# 5. Plot and find the best threshold.

# Your code here...

print(f"Best Threshold found: {best_threshold:.2f} with Dice: {max_dice:.4f}")

## Block 9.2: Reading the Errors of a Segmentation Model
### Why an average is not enough

An average Dice of 0.80 on validation can be composed in many different ways: 100 images with Dice 0.80 each, or 90 images with Dice 0.95 and 10 images with Dice 0.05. The average does not distinguish these two scenarios, but clinically they are radically different — the second case means there is a subgroup of patients where the model fails completely, hidden behind a good overall average.

### TP, TN, FP and FN at pixel level

So far you have computed Dice and IoU as formulas over sets of pixels, without explicitly naming their four components. In segmentation, each pixel of the predicted mask falls into one of these four categories when compared with the ground truth:

| | GT = tumor | GT = background |
|---|---|---|
| **Prediction = tumor** | TP (hit) | FP (false alarm) |
| **Prediction = background** | FN (miss, it does not detect it) | TN (hit) |

In a **normal** image (real mask completely empty, without a single tumor pixel), it is mathematically impossible to have TP or FN — there is no tumor to hit or to miss. The only possible outcomes are TN (all the background correctly ignored) or FP (the model "hallucinates" a tumor where none exists). That is why Case 1 of this task (a normal image) is the most direct and clean check of whether your model produces False Positives.

### Why choosing the cases on purpose, and not at random

The BUSI dataset is imbalanced (437 benign, 210 malignant, 133 normal). If you chose 5 images purely at random from the `val_df`, most likely all 5 would be `benign` — the majority class — and you would never see how the model behaves with `normal` or `malignant` images. It is the same problem as the split in Block 7.1, now applied to visual inspection: a random sampling blind to the minority classes teaches you nothing about them. That is why the task explicitly asks for one case of each class, and not "any 5 images".

### How to find the "blatant failures" without relying on luck

For Cases 4 and 5, the instructions allow searching by "iterating" instead of at random. The systematic way to do it is: compute the individual Dice of each image of `val_df` (not the average, one per image),
and sort from lowest to highest. The images with the lowest Dice are, by definition, those with the most errors — finding them by brute force is much more reliable than trusting that they will be picked by chance in a small random sample.

### What to look at in the overlay

The overlay (prediction in red over the original image) is where the *type* of error is seen, not just whether there was an error:

| Visual pattern | Error type | Effect on Dice |
|-----------------|-----------------|-------------------|
| Red falls short of the real tumor contour | Under-segmentation (FN at the border) | Reduces Dice |
| Red extends beyond the real contour | Over-segmentation (FP at the border) | Reduces Dice |
| Red appears over healthy tissue with no tumor nearby | "Hallucinated" False Positive | Reduces Dice and is clinically serious |
| There is no red where there is a tumor | Total False Negative | The most serious error clinically |

### From technical to clinical

The checkpoint does not only ask to identify TP/TN/FP/FN — it asks to explain **what each type of error means clinically**. It is not the same to say "there are FP pixels at the upper border" as to say "the model would mark a healthy area as suspicious, which would lead to unnecessary tests or biopsies in this patient". The second sentence is the one that connects with the FP/FN dilemma of the introduction of this practice.

### Quick summary

| Concept | Main idea |
|----------|-----------------|
| Average vs individual cases | A good average Dice can hide serious failures in specific subgroups |
| TP/TN/FP/FN per pixel | In normal images only TN or FP are possible — never TP or FN |
| Deliberate selection | Random sampling in imbalanced data hides the minority classes |
| Finding failures | Sort by individual Dice, do not rely on chance |
| Overlay | The visual pattern (missing, extra, or appearing where it should not) indicates the error type |
| Clinical explanation | Translate TP/FP/FN into real consequences for the patient |

## Task 9.2: Qualitative Error Analysis
Numeric metrics do not tell the whole story. A Dice of 0.80 can hide serious errors in clinically relevant cases. Let us visualize **where** the model is failing.

### Instructions:
1. Manually (or randomly) select 5 images from the `val_df` that represent:
   * **Case 1:** A **"normal"** class image (to check if there are False Positives).
   * **Case 2:** A **"benign"** class image with a small tumor.
   * **Case 3:** A **"malignant"** class image with irregular borders.
   * **Cases 4 and 5:** Two images where the model failed catastrophically (you can look for them by iterating if you want, or use random ones).
2. For each one, generate a figure with 4 columns:
   * **Original**
   * **Ground Truth (Real Mask)**
   * **Prediction** (using your `best_threshold`)
   * **Overlay** (Prediction in red over the Original)
3. Add a title to each image indicating its class and whether it is TP, TN, FP or FN.

> **CHECKPOINT P9.1:** Show the professor the 5 images and clinically explain what is happening in the errors.

In [ ]:
# ============================================================
# TASK 9.2: QUALITATIVE ERROR ANALYSIS
# ============================================================
import cv2

# WRITE YOUR CODE HERE
# 1. Select 5 indices of val_df (you can use val_df.sample(5) or choose them by hand)
# 2. Create a figure plt.subplots(5, 4, figsize=(16, 20))
# 3. Loop for each image:
#       - Load the image and the real mask
#       - Pass the image through the model (preprocessing the same as in the Dataset)
#       - Compute the prediction with best_threshold
#       - Show the 4 columns (Original, GT, Pred, Overlay)

# Your code here...

plt.tight_layout()
plt.show()

## Block 9.3: Closing the Loop — From Calibration to Submission
### The same type of pipeline you already built in Practice 7

This task asks you to conceptually repeat the same inference pipeline of Block 7.4: load a test image, preprocess it the same as in training, pass it through the model, threshold it, resize the mask to the original size, convert to RLE and save the CSV. The only thing that changes is **which threshold value you use to binarize**: before it was a fixed `0.5`, now it is your `best_threshold` calibrated in Task 1.

> **Key idea:** if you find yourself rewriting all the preprocessing from scratch, stop for a moment — it is practically the same code you already wrote in Practice 7, with a single different value.

### Closing the threshold loop

In Block 9.1 you calibrated `best_threshold` using the **validation** set — precisely because you could not use train (biased) nor test (you do not have its masks to measure Dice there). Now, in Task 9.3, that same fixed value is applied to the **test** set, without recomputing it. This is the complete and correct cycle of a hyperparameter: it is calibrated once on validation, and that frozen value is the one deployed on new data.

> **Key idea:** if at this point you found yourself recomputing the threshold on the test itself, you would be repeating exactly the error that Block 9.1 explained to you why to avoid.

### A technical nuance before resizing the mask

The model always predicts at `IMG_SIZE` resolution, but each test image has its own original size — so at some point you will have to resize the predicted mask back to `(original_h, original_w)`, just like in Practice 7.

Here there is a detail worth being clear about **before** writing the code, because it is an easy bug to make without any error being raised: if you resize a mask that is **already binary** (only zeros and ones) with `cv2.resize`, the default interpolation can mix values at the tumor borders, leaving pixels with intermediate values (neither 0 nor 1):

```
small binary mask (IMG_SIZE)  →  cv2.resize  →  no longer purely 0/1 at the borders
```

You have two valid ways to avoid it:

1. **Binarize, resize, and binarize again**: you apply the threshold at `IMG_SIZE`, resize that mask, and apply a second cut (for example `> 0.5`) after the resize to clean the intermediate values introduced by the interpolation.
2. **Resize the probabilities, binarize only once**: you resize the continuous output of `sigmoid()` (not the binary mask) to the original size, and apply `best_threshold` a single time, already at the final resolution.

Either one is correct. What is **not** correct is resizing a binary mask and passing it directly to `mask_to_rle` without that second cut — the resulting RLE would have slightly different tumor boundaries than what your model actually predicted, without anything in the execution warning you about the problem.

### Why a different file name

The task asks to save `submission_final.csv`, not to overwrite the `submission.csv` from Practice 7. Keeping different names per version allows you to compare on the Kaggle Leaderboard what concrete improvement each step brought (baseline → augmentation → optimal threshold), instead of losing that traceability by overwriting the same file every time.

### The complete pipeline, at a glance

```
P7: Trained model (fixed threshold 0.5)
        │
P8: Does augmentation improve?  →  final model
        │
Block 9.1: best_threshold calibrated on validation
        │
Task 9.3: final model + best_threshold  →  submission_final.csv
```

### Quick summary

| Concept | Main idea |
|----------|-----------------|
| Reuse | The pipeline is conceptually the one from Block 7.4; only the threshold used changes |
| Hyperparameter cycle | It is calibrated once on validation, applied fixed on test |
| Binarize vs resize | Resizing an already binary mask reintroduces intermediate values; you must re-binarize or resize the probabilities before thresholding |
| File naming | Versioning the submissions allows attributing the improvement to each concrete change |

## Task 9.3: Final Submission Generation
The moment of truth has arrived. We are going to apply everything we have learned (best model, best threshold) to the test set to generate our definitive `submission.csv`.

### Instructions:
1. Iterate over all the images in the `test/images` folder.
2. Preprocess each image exactly the same as in Task 9.2.
3. Predict the mask using the **`best_threshold`** you found in Task 9.1.
4. Resize the mask to the original size of the image.
5. Convert the mask to **RLE** format (using the `mask_to_rle` function from P6).
6. Save the results in a DataFrame and export it as **`submission_final.csv`**.

> **CHECKPOINT P9.2:** Download the `submission_final.csv`, upload it to Kaggle and show the professor your new score on the Leaderboard. Go for the top 10!

In [ ]:
# ============================================================
# TASK 9.3: FINAL SUBMISSION GENERATION
# ============================================================
import os
import pandas as pd

# Make sure you have the mask_to_rle function defined (copy it from P6 if necessary)
# def mask_to_rle(mask): ...

test_img_dir = DATA_DIR / 'test' / 'images'
test_images = sorted(list(test_img_dir.glob('*.png')))

results = []

print(f" Generating final submission with threshold = {best_threshold:.2f}...")
model.eval()

with torch.no_grad():
    for idx, img_path in enumerate(test_images):
        # 1. Load and preprocess (same as in Task 9.2)
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        original_h, original_w = img_rgb.shape[:2]

        # ... (Your preprocessing code here) ...

        # 2. Predict with best_threshold
        # ... (Your prediction code here) ...

        # 3. Convert to RLE and save
        rle = mask_to_rle(mask_binary)
        results.append({'Id': img_path.name, 'Expected': rle})

        if (idx + 1) % 50 == 0:
            print(f"   Progress: {idx + 1}/{len(test_images)} images")

# Create DataFrame and save
submission_df = pd.DataFrame(results)
submission_df.to_csv('submission_final.csv', index=False)
print("submission_final.csv generated successfully. Upload it to Kaggle!")